# Control demo de vacaciones y permisos — MariaDB en Colab

Este cuaderno instala MariaDB en la máquina temporal de Colab, crea `rrhh_demo`, carga `sql/schema.sql` y permite consultar, registrar y revisar solicitudes ficticias. Si el entorno se reinicia, vuelve a ejecutar las celdas de instalación y carga. No uses datos reales de trabajadores.

## 1. Instalar MariaDB y preparar los archivos

Ejecuta esta celda. Si ya clonaste el repositorio durante esta sesión, lo actualiza para usar la última versión.

In [ ]:
from pathlib import Path
import subprocess
import os

repo = Path('/content/Municipalidad-Colab')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/GimAlvieri/Municipalidad-Colab.git', str(repo)], check=True)

!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y mariadb-server nodejs npm
!service mariadb start
print('SQL del proyecto:', repo / 'sql' / 'schema.sql')

## 2. Crear la base y cargar las tablas

La contraseña `colab_demo` solo es para esta práctica con datos ficticios. La base se crea dentro de Colab y es temporal.

In [ ]:
!mysql -u root -e "CREATE DATABASE IF NOT EXISTS rrhh_demo CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci; CREATE USER IF NOT EXISTS 'rrhh_app'@'localhost' IDENTIFIED BY 'colab_demo'; CREATE USER IF NOT EXISTS 'rrhh_app'@'127.0.0.1' IDENTIFIED BY 'colab_demo'; GRANT ALL PRIVILEGES ON rrhh_demo.* TO 'rrhh_app'@'localhost'; GRANT ALL PRIVILEGES ON rrhh_demo.* TO 'rrhh_app'@'127.0.0.1'; FLUSH PRIVILEGES;"
!mysql -u root rrhh_demo < /content/Municipalidad-Colab/sql/schema.sql
!mysql -u root rrhh_demo -e "SHOW TABLES; SELECT * FROM resumen_por_estado ORDER BY area, estado, tipo;"

## 3. Conectarse desde Python y ver los reportes

Esta conexión va a MariaDB en esta sesión de Colab; no requiere Supabase ni una URL externa.

In [ ]:
%pip -q install mysql-connector-python pandas ipywidgets
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(host='127.0.0.1', port=3306, user='rrhh_app',
                               password='colab_demo', database='rrhh_demo')
cur = conn.cursor(dictionary=True)
cur.execute('SELECT area, estado, tipo, cantidad FROM resumen_por_estado ORDER BY area, estado, tipo')
resumen = pd.DataFrame(cur.fetchall())
display(resumen)

if not resumen.empty:
    grafico = resumen.groupby('estado')['cantidad'].sum().sort_values(ascending=False)
    ax = grafico.plot(kind='bar', title='Solicitudes ficticias por estado',
                      xlabel='Estado', ylabel='Cantidad', color='#179b83', rot=0)
    ax.figure.tight_layout()

In [ ]:
cur.execute('''SELECT s.id, s.codigo_trabajador, a.nombre AS area, s.tipo,
                     s.fecha_inicio, s.fecha_fin, s.estado
              FROM solicitudes s JOIN areas a ON a.id = s.area_id
              ORDER BY s.fecha_inicio''')
display(pd.DataFrame(cur.fetchall()))

## 4. Registrar una solicitud ficticia

Usa un código `DEMO-...`; no escribas nombres, DNI ni datos reales.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

cur.execute('SELECT id, nombre FROM areas ORDER BY nombre')
areas = cur.fetchall()
area_widget = widgets.Dropdown(options=[(a['nombre'], a['id']) for a in areas], description='Área:')
codigo_widget = widgets.Text(value='DEMO-004', description='Código:')
tipo_widget = widgets.Dropdown(options=['vacaciones', 'permiso'], description='Tipo:')
inicio_widget = widgets.DatePicker(description='Desde:')
fin_widget = widgets.DatePicker(description='Hasta:')
guardar_widget = widgets.Button(description='Registrar', button_style='primary')
salida_widget = widgets.Output()

def registrar(_):
    with salida_widget:
        clear_output()
        codigo = codigo_widget.value.strip().upper()
        if not codigo.startswith('DEMO-'):
            print('Usa un código ficticio que comience con DEMO-.')
            return
        if inicio_widget.value is None or fin_widget.value is None or fin_widget.value < inicio_widget.value:
            print('Selecciona fechas válidas; la fecha final debe ser igual o posterior a la inicial.')
            return
        cur.execute('''INSERT INTO solicitudes
            (codigo_trabajador, area_id, tipo, fecha_inicio, fecha_fin, estado, observacion_demo)
            VALUES (%s, %s, %s, %s, %s, 'pendiente', 'Registro ficticio desde Colab')''',
            (codigo, area_widget.value, tipo_widget.value, inicio_widget.value, fin_widget.value))
        conn.commit()
        print('Solicitud ficticia registrada.')

guardar_widget.on_click(registrar)
display(widgets.VBox([codigo_widget, area_widget, tipo_widget, inicio_widget, fin_widget, guardar_widget, salida_widget]))

## 5. Iniciar la interfaz HTML/CSS (opcional)

Instala las dependencias de Node e inicia el servidor conectado a la MariaDB de Colab. El enlace solo funciona mientras esta sesión esté activa.

In [ ]:
import time
import subprocess
import os
from IPython.display import HTML, display
from google.colab import output

subprocess.run(['npm', 'install'], cwd=repo, check=True)
web_env = os.environ.copy()
web_env.update({'DB_HOST': '127.0.0.1', 'DB_PORT': '3306', 'DB_NAME': 'rrhh_demo',
                 'DB_USER': 'rrhh_app', 'DB_PASSWORD': 'colab_demo',
                 'HOST': '0.0.0.0', 'PORT': '3000'})
server_log = open('/tmp/rrhh-demo-server.log', 'w')
server_process = subprocess.Popen(['node', 'server.js'], cwd=repo, env=web_env,
                                  stdout=server_log, stderr=server_log)
time.sleep(3)
if server_process.poll() is not None:
    print(Path('/tmp/rrhh-demo-server.log').read_text())
else:
    web_url = output.eval_js('google.colab.kernel.proxyPort(3000)')
    display(HTML(f'<a href="{web_url}" target="_blank">Abrir la aplicación de Recursos Humanos</a>'))

## Cerrar la conexión Python

Ejecuta esta celda al terminar. Para recrear la base después de reiniciar Colab, vuelve a correr las secciones 1 y 2.

In [ ]:
cur.close()
conn.close()
print('Conexión cerrada.')